In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

# Simulate embeddings learned by a Siamese network
np.random.seed(42)
n_samples = 100
embedding_dim = 2  # For easy visualization

# Simulate data points belonging to two classes (0 and 1)
labels = np.random.randint(0, 2, n_samples)

# Generate embeddings such that points from the same class are closer
embeddings = np.random.randn(n_samples, embedding_dim)
embeddings[labels == 0] += np.array([-1, -1])
embeddings[labels == 1] += np.array([1, 1])
embeddings += 0.5 * np.random.randn(n_samples, embedding_dim) # Add some noise

# ---------------------- Contrastive Loss Visualization ----------------------

# Simulate pairs for contrastive loss
pairs_contrastive = []
pair_labels_contrastive = []
for i in range(n_samples):
    for j in range(i + 1, n_samples):
        pairs_contrastive.append((embeddings[i], embeddings[j]))
        pair_labels_contrastive.append(1 if labels[i] == labels[j] else 0) # 1 for similar, 0 for dissimilar

pairs_contrastive = np.array(pairs_contrastive)
pair_labels_contrastive = np.array(pair_labels_contrastive)

# Imagine the network tries to minimize the distance for similar pairs and maximize for dissimilar
# We'll project the embeddings to 1D to simulate the loss effect on distance

def project_to_1d(embeddings, axis=0):
    return embeddings[:, axis]

proj_embeddings_c = project_to_1d(embeddings)

# Separate projections for similar and dissimilar pairs (just the first dimension for simplicity)
similar_distances_c = []
dissimilar_distances_c = []
for i in range(len(pairs_contrastive)):
    emb1_proj = project_to_1d(np.array([pairs_contrastive[i][0]]))[0]
    emb2_proj = project_to_1d(np.array([pairs_contrastive[i][1]]))[0]
    distance = np.abs(emb1_proj - emb2_proj)
    if pair_labels_contrastive[i] == 1:
        similar_distances_c.append(distance)
    else:
        dissimilar_distances_c.append(distance)

# ------------------------ Triplet Loss Visualization ------------------------

# Simulate triplets for triplet loss (anchor, positive, negative)
triplets = []
for i in range(n_samples):
    anchor = embeddings[i]
    positive_indices = np.where(labels == labels[i])[0]
    negative_indices = np.where(labels != labels[i])[0]

    if len(positive_indices) > 1 and len(negative_indices) > 0:
        positive_index = np.random.choice(positive_indices[positive_indices != i])
        negative_index = np.random.choice(negative_indices)
        triplets.append((anchor, embeddings[positive_index], embeddings[negative_index]))

triplets = np.array(triplets)

# Imagine the network tries to pull positive closer to anchor and push negative further
# Again, project to 1D to simulate the loss effect on distance

proj_embeddings_t = project_to_1d(embeddings)

anchor_positive_distances_t = []
anchor_negative_distances_t = []
for anchor, positive, negative in triplets:
    anchor_proj = project_to_1d(np.array([anchor]))[0]
    positive_proj = project_to_1d(np.array([positive]))[0]
    negative_proj = project_to_1d(np.array([negative]))[0]
    anchor_positive_distances_t.append(np.abs(anchor_proj - positive_proj))
    anchor_negative_distances_t.append(np.abs(anchor_proj - negative_proj))

# ------------------------------- Plotting -------------------------------

plt.figure(figsize=(14, 6))

# Subplot 1: Embeddings in 2D
plt.subplot(1, 2, 1)
scatter = plt.scatter(embeddings[:, 0], embeddings[:, 1], c=labels, cmap='viridis')
plt.title('Simulated Embeddings (2D)')
plt.xlabel('Embedding Dimension 1')
plt.ylabel('Embedding Dimension 2')
legend = plt.legend(*scatter.legend_elements(), title="Classes")
plt.gca().add_artist(legend)
plt.grid(True)

# Subplot 2: Distribution of Distances (Simulated Loss Effect)
plt.subplot(1, 2, 2)

# Contrastive Loss
plt.hist(similar_distances_c, alpha=0.5, label='Similar Pair Distances (Contrastive)', color='blue')
plt.hist(dissimilar_distances_c, alpha=0.5, label='Dissimilar Pair Distances (Contrastive)', color='red')

# Triplet Loss (shifted for better visualization)
plt.hist(np.array(anchor_positive_distances_t) + 0.2, alpha=0.5, label='Anchor-Positive Distances (Triplet)', color='lightgreen')
plt.hist(np.array(anchor_negative_distances_t) - 0.2, alpha=0.5, label='Anchor-Negative Distances (Triplet)', color='salmon')

plt.xlabel('Simulated Distance in Embedding Space (1D Projection)')
plt.ylabel('Frequency')
plt.title('Simulated Effect of Contrastive vs. Triplet Loss on Distances')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()
